In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

# =========================================================
# IMDb RESULTS ANALYSIS — CORRECTED VERSION
# =========================================================
# This notebook reproduces the same output variables and artifacts
# as the previous analysis notebook, but now aligned with the paper:
#
# 1) Metrics are computed per (query, scale factor, run phase)
# 2) Top-1 preservation checks whether the best configuration in the
#    full benchmarked space belongs to the activated family
# 3) Near-best preservation (5% threshold) is added
#
# Expected input:
# - benchmark_aggregate_results.csv
#
# Main outputs:
# - analysis_df with:
#   official_id, query_name, scale_label, run_phase,
#   n_tested_configs, n_activated_configs, DSR,
#   best_config, best_group, best_design_pattern, best_p95_ms,
#   top1_preserved_by_activated, near_best_preserved_by_activated,
#   activated_regret,
#   best_primary_config, best_primary_p95_ms, primary_regret
# - global summary metrics
# - a filtered table containing only rows whose best overall group is
#   secondary_affected
# - exported CSV artifacts in the output folder
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------
results_dir = Path("/home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/results/sf_050")   # change if needed
results_csv = results_dir / "benchmark_aggregate_results.csv"
run_phase_to_analyze = "hot"

# Output folder for exported artifacts
output_dir = results_dir / "imdb_analysis_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# IMDb broader tested configuration space:
# G0, G2, G3, G4, G5, G6, G7, G8, G9 = 9
n_C = 9

# Near-best threshold used in the paper
near_best_threshold = 0.05

# ---------------------------------------------------------
# 2) Read benchmark aggregate results
# ---------------------------------------------------------
agg = pd.read_csv(results_csv)

required_cols = [
    "config_name",
    "activated_class",
    "benchmark_family",
    "scale_label",
    "query_name",
    "query_group",
    "run_phase",
    "p95_latency_ms",
]
missing = [c for c in required_cols if c not in agg.columns]
if missing:
    raise ValueError(f"Missing required columns in benchmark_aggregate_results.csv: {missing}")

# ---------------------------------------------------------
# 3) Derive official query id
# ---------------------------------------------------------
# Examples:
#   QG1_WatchItemById -> QG1
#   QG10_AdvancedSearchWatchItems -> QG10
agg["official_id"] = agg["query_name"].str.extract(r"^(QG\d+)")

# ---------------------------------------------------------
# 4) Restrict to the selected run phase
# ---------------------------------------------------------
phase_df = agg[agg["run_phase"] == run_phase_to_analyze].copy()

if phase_df.empty:
    raise ValueError(f"No rows found for run_phase = {run_phase_to_analyze!r}")

# ---------------------------------------------------------
# 5) Build the query-level summary table
# ---------------------------------------------------------
rows = []

group_cols = ["query_name", "scale_label", "run_phase"]

for (query_name, scale_label, run_phase), grp in phase_df.groupby(group_cols):
    grp = grp.copy()

    # Best configuration in the full tested space
    best_all = grp.loc[grp["p95_latency_ms"].idxmin()]
    best_latency_full = float(best_all["p95_latency_ms"])
    best_config_full = str(best_all["activated_class"])

    # Activated set = non-control rows
    activated = grp[grp["query_group"] != "control"].copy()
    activated_classes = set(activated["activated_class"].dropna().astype(str).unique())

    n_tested_configs = grp["activated_class"].nunique()
    n_activated_configs = activated["activated_class"].nunique()
    dsr = 1 - (n_activated_configs / n_C)

    # Top-1 preservation:
    # the best configuration from the full space belongs to the activated family
    top1_preserved_by_activated = best_config_full in activated_classes

    # Near-best preservation:
    # at least one activated configuration is within 5% of the best full-space latency
    near_best_mask = (
        (grp["p95_latency_ms"] - best_latency_full) / best_latency_full
    ) <= near_best_threshold
    near_best_classes = set(
        grp.loc[near_best_mask, "activated_class"].dropna().astype(str).unique()
    )
    near_best_preserved_by_activated = (
        len(activated_classes.intersection(near_best_classes)) > 0
    )

    # Activated regret
    if activated.empty:
        activated_regret = np.nan
    else:
        best_activated = activated.loc[activated["p95_latency_ms"].idxmin()]
        activated_regret = (
            float(best_activated["p95_latency_ms"]) - best_latency_full
        ) / best_latency_full

    # Best primary-only configuration
    primary = grp[grp["query_group"] == "primary"].copy()

    if primary.empty:
        best_primary_config = None
        best_primary_p95_ms = np.nan
        primary_regret = np.nan
    else:
        best_primary = primary.loc[primary["p95_latency_ms"].idxmin()]
        best_primary_config = best_primary["activated_class"]
        best_primary_p95_ms = float(best_primary["p95_latency_ms"])
        primary_regret = (best_primary_p95_ms - best_latency_full) / best_latency_full

    rows.append({
        "official_id": best_all["official_id"],
        "query_name": query_name,
        "scale_label": scale_label,
        "run_phase": run_phase,

        "n_tested_configs": int(n_tested_configs),
        "n_activated_configs": int(n_activated_configs),
        "DSR": float(dsr),

        "best_config": best_all["activated_class"],
        "best_group": best_all["query_group"],
        "best_design_pattern": best_all["benchmark_family"],
        "best_p95_ms": best_latency_full,

        "top1_preserved_by_activated": bool(top1_preserved_by_activated),
        "near_best_preserved_by_activated": bool(near_best_preserved_by_activated),
        "activated_regret": float(activated_regret) if pd.notna(activated_regret) else np.nan,

        "best_primary_config": best_primary_config,
        "best_primary_p95_ms": best_primary_p95_ms,
        "primary_regret": float(primary_regret) if pd.notna(primary_regret) else np.nan,
    })

analysis_df = pd.DataFrame(rows).sort_values(
    by=["official_id", "scale_label", "run_phase"]
).reset_index(drop=True)

print(f"IMDb query-level summary using run_phase = {run_phase_to_analyze!r}\n")
display(analysis_df)

# ---------------------------------------------------------
# 6) Global summary metrics
# ---------------------------------------------------------
print("Average DSR:", analysis_df["DSR"].mean())
print("Top-1 preservation activated:", analysis_df["top1_preserved_by_activated"].mean())
print("Near-best preservation activated:", analysis_df["near_best_preserved_by_activated"].mean())
print("Mean activated regret:", analysis_df["activated_regret"].dropna().mean())
print("Mean primary regret:", analysis_df["primary_regret"].dropna().mean())

# ---------------------------------------------------------
# 7) Secondary-affected winners only
# ---------------------------------------------------------
secondary_winners_df = analysis_df[
    analysis_df["best_group"] == "secondary_affected"
][
    [
        "official_id",
        "query_name",
        "scale_label",
        "best_config",
        "best_design_pattern",
        "best_p95_ms",
        "best_primary_config",
        "best_primary_p95_ms",
        "primary_regret",
    ]
].reset_index(drop=True)

print()
display(secondary_winners_df)

# ---------------------------------------------------------
# 8) Additional artifact: best configuration by query and scale
# ---------------------------------------------------------
best_by_query_scale_rows = []

for (query_name, scale_label), grp in phase_df.groupby(["query_name", "scale_label"]):
    best = grp.loc[grp["p95_latency_ms"].idxmin()]
    best_by_query_scale_rows.append({
        "official_id": best["official_id"],
        "query_name": query_name,
        "scale_label": scale_label,
        "best_config": best["activated_class"],
        "best_group": best["query_group"],
        "best_design_pattern": best["benchmark_family"],
        "best_p95_ms": best["p95_latency_ms"],
        "best_avg_latency_ms": best.get("avg_latency_ms", np.nan),
        "best_p99_ms": best.get("p99_latency_ms", np.nan),
        "avg_documents_returned": best.get("avg_documents_returned", np.nan),
    })

best_by_query_scale_df = pd.DataFrame(best_by_query_scale_rows).sort_values(
    by=["official_id", "scale_label"]
).reset_index(drop=True)

# ---------------------------------------------------------
# 9) Export artifacts
# ---------------------------------------------------------
analysis_df.to_csv(output_dir / f"imdb_summary_{run_phase_to_analyze}_by_scale.csv", index=False)
secondary_winners_df.to_csv(output_dir / f"imdb_diff_best_vs_primary_{run_phase_to_analyze}_by_scale.csv", index=False)
best_by_query_scale_df.to_csv(output_dir / f"imdb_best_by_query_scale_{run_phase_to_analyze}.csv", index=False)

print(f"\nSaved outputs in: {output_dir.resolve()}")


IMDb query-level summary using run_phase = 'hot'



,official_id,query_name,scale_label,run_phase,n_tested_configs,n_activated_configs,DSR,best_config,best_group,best_design_pattern,best_p95_ms,top1_preserved_by_activated,near_best_preserved_by_activated,activated_regret,best_primary_config,best_primary_p95_ms,primary_regret
0,QG1,QG1_WatchItemById,sf0.5,hot,9,6,0.333333,G7,control,containment_family,0.192424,False,False,0.311679,G0,0.270370,0.405073
1,QG10,QG10_AdvancedSearchWatchItems,sf0.5,hot,9,9,0.000000,G7,secondary_affected,containment_family,171.641413,True,True,0.000000,G3,174.507078,0.016696
2,QG2,QG2_WatchItemByTitle,sf0.5,hot,9,6,0.333333,G7,control,containment_family,0.285235,False,False,0.074828,G0,0.306578,0.074828
3,QG3,QG3_RecommendationByGenreAndSubtype,sf0.5,hot,9,9,0.000000,G8,secondary_affected,containment_family,7.588797,True,True,0.000000,G0,7.950161,0.047618
4,QG4,QG4_AllPersonsOfTypeForWatchItem,sf0.5,hot,9,6,0.333333,G5,primary,associative_family,0.333034,True,True,0.000000,G5,0.333034,0.000000
5,QG5,QG5_AllPersonsForEpisodesOfSeries,sf0.5,hot,9,9,0.000000,G4,primary,associative_family,164.455528,True,True,0.000000,G4,164.455528,0.000000
6,QG6,QG6_EpisodesOfSeries,sf0.5,hot,9,4,0.555556,G7,primary,containment_family,1.016240,True,True,0.000000,G7,1.016240,0.000000
7,QG7,QG7_UpdateWatchItemMetadata,sf0.5,hot,9,6,0.333333,G7,control,containment_family,0.218918,False,False,0.063108,G0,0.384026,0.754197
8,QG8,QG8_AddPersonRoleToWatchItem,sf0.5,hot,9,6,0.333333,G7,control,containment_family,0.358465,False,True,0.049870,G0,0.401465,0.119956
9,QG9,QG9_TopRatedSeriesByGenre,sf0.5,hot,9,9,0.000000,G7,secondary_affected,containment_family,2.054243,True,True,0.000000,G2,78.136428,37.036610


Average DSR: 0.22222222222222224
Top-1 preservation activated: 0.6
Near-best preservation activated: 0.7
Mean activated regret: 0.049948545650370234
Mean primary regret: 3.8454977133328767



,official_id,query_name,scale_label,best_config,best_design_pattern,best_p95_ms,best_primary_config,best_primary_p95_ms,primary_regret
0,QG10,QG10_AdvancedSearchWatchItems,sf0.5,G7,containment_family,171.641413,G3,174.507078,0.016696
1,QG3,QG3_RecommendationByGenreAndSubtype,sf0.5,G8,containment_family,7.588797,G0,7.950161,0.047618
2,QG9,QG9_TopRatedSeriesByGenre,sf0.5,G7,containment_family,2.054243,G2,78.136428,37.036610



Saved outputs in: /home/jovyan/privado/framework evaluation approachs/framework with dataset imdb oficial/results/sf_050/imdb_analysis_outputs
